# HT Storage Processor

Auto-detects NDA / .xls / .xlsx files. Generates per-lot and per-cell charts with readable x-axis labels, plus a stage1ready.pkl cache.


# 1. Imports + pickers

In [ ]:
import os, re
import pandas as pd
import numpy as np
import traitlets
from ipywidgets import widgets
from IPython.display import display, clear_output
import warnings; warnings.filterwarnings("ignore")

try:
    import NewareNDA
    HAS_NDA = True
except Exception:
    HAS_NDA = False
    print("(NewareNDA not installed)")

def _pick_dir():
    try:
        import tkinter as tk; from tkinter import filedialog
        root = tk.Tk(); root.withdraw()
        try: root.call("wm","attributes",".","-topmost",True)
        except Exception: pass
        d = filedialog.askdirectory()
        try: root.update(); root.destroy()
        except Exception: pass
        return d or None
    except Exception as e:
        print(f"  (Tk failed: {e}; type path manually)"); return None

def _pick_file():
    try:
        import tkinter as tk; from tkinter import filedialog
        root = tk.Tk(); root.withdraw()
        try: root.call("wm","attributes",".","-topmost",True)
        except Exception: pass
        p = filedialog.askopenfilename()
        try: root.update(); root.destroy()
        except Exception: pass
        return p or None
    except Exception as e:
        print(f"  (Tk failed: {e}; paste path)"); return None

class _DirBtn(widgets.Button):
    def __init__(self, label, target):
        super().__init__()
        self.description=label; self.icon="folder-open-o"; self.style.button_color="orange"
        self._t=target; self.layout=widgets.Layout(width="280px")
        self.on_click(self._click)
    def _click(self, b):
        with _out:
            clear_output(); d=_pick_dir()
            if d:
                self._t.value=d; b.style.button_color="lightgreen"; b.description="Selected"; print(d)

class _FileBtn(widgets.Button):
    def __init__(self, label, target):
        super().__init__()
        self.description=label; self.icon="file-excel-o"; self.style.button_color="orange"
        self._t=target; self.layout=widgets.Layout(width="280px")
        self.on_click(self._click)
    def _click(self, b):
        with _out:
            clear_output(); p=_pick_file()
            if p:
                self._t.value=p; b.style.button_color="lightgreen"; b.description="Selected"; print(p)

_out = widgets.Output()

def _row(lbl, btn_lbl):
    txt = widgets.Text(description=lbl+":", layout=widgets.Layout(width="700px"),
                       placeholder="Click button or paste path")
    return widgets.HBox([_DirBtn(btn_lbl, txt), txt]), txt

w_for, t_for = _row("Formation",      "Pick Formation dir")
w_bef, t_bef = _row("Before Storage", "Pick Before Storage dir")
w_2w,  t_2w  = _row("After 2 Weeks",  "Pick 2W dir")
w_1m,  t_1m  = _row("After 1 Month",  "Pick 1M dir")

t_meta = widgets.Text(description="metadata xlsx:", layout=widgets.Layout(width="900px"),
                      placeholder="Barcode -> Group/Electrolyte mapping")
b_meta = _FileBtn("Pick metadata", t_meta)

# OPTIONAL — thickness/ACR xlsx (skip if not measured)
t_thick = widgets.Text(description="thickness xlsx:", layout=widgets.Layout(width="900px"),
                       placeholder="(optional) HT storage thickness.xlsx — leave empty to skip")
b_thick = _FileBtn("Pick thickness xlsx", t_thick)

t_out_dir = widgets.Text(description="output folder:", layout=widgets.Layout(width="900px"))
b_out_dir = _DirBtn("Pick output dir", t_out_dir)
t_fname = widgets.Text(value="HT_storage_output.xlsx", description="filename:",
                       layout=widgets.Layout(width="500px"))

display(widgets.VBox([
    widgets.HTML("<b>Time-point folders</b> (leave empty to skip):"),
    w_for, w_bef, w_2w, w_1m,
    widgets.HTML("<b>Metadata + output</b>:"),
    widgets.HBox([b_meta, t_meta]),
    widgets.HBox([b_thick, t_thick]),
    widgets.HBox([b_out_dir, t_out_dir]),
    t_fname, _out
]))


# 2. Step indices

In [ ]:
step_idx = {
    "for":    {"cap_full_disch_step": 4, "rest_before_dcir": 11, "dcir_pulse_step": 12},
    "before": {"cap_full_disch_step": 4, "cap_full_chg_step": 14, "rest_before_dcir": 9, "dcir_pulse_step": 10},
    "after":  {"cap_cyc1_disch_step": 2, "cap_cyc2_chg_step": 4, "cap_cyc2_disch_step": 6,
               "rest_before_dcir": 11, "dcir_pulse_step": 12},
}
print("Step indices:", step_idx)


# 3. Helpers (auto-detect column convention)

In [ ]:
def detect_format(path):
    e = os.path.splitext(path)[1].lower()
    if e in (".nda", ".ndax"): return "nda"
    if e == ".xls": return "xls"
    if e in (".xlsx", ".xlsm"): return "xlsx"
    return None

def _normalize_neware_old(df):
    out = pd.DataFrame()
    out["Step Number"]    = df.get("Step", df.get("Step Index"))
    out["Step Type"]      = df.get("Status", df.get("Step Type", ""))
    chg = pd.to_numeric(df.get("Charge_Capacity(mAh)", 0), errors="coerce").fillna(0)
    dch = pd.to_numeric(df.get("Discharge_Capacity(mAh)", 0), errors="coerce").fillna(0)
    out["Capacity(Ah)"]   = (chg + dch) / 1000.0
    out["Oneset Volt.(V)"] = df.get("Start Voltage(V)", df.get("Oneset Volt.(V)", np.nan))
    out["End Voltage(V)"]  = df["End Voltage(V)"]
    i_col = next((c for c in ["Start Current(mA)","Starting current(mA)","Starting current(A)","End Current(mA)"] if c in df.columns), None)
    if i_col is None:
        out["Starting current(A)"] = np.nan
    else:
        v = pd.to_numeric(df[i_col], errors="coerce")
        out["Starting current(A)"] = v / 1000.0 if "mA" in i_col else v
    return out

def _normalize_nw_export(df):
    out = pd.DataFrame()
    out["Step Number"]    = df.get("Step Number", df.get("Step Index", df.get("Step")))
    out["Step Type"]      = df.get("Step Type", df.get("Status", ""))
    out["Capacity(Ah)"]   = pd.to_numeric(df["Capacity(Ah)"], errors="coerce")
    out["Oneset Volt.(V)"] = df.get("Oneset Volt.(V)", df.get("Start Voltage(V)", np.nan))
    out["End Voltage(V)"]  = df["End Voltage(V)"]
    out["Starting current(A)"] = pd.to_numeric(df.get("Starting current(A)", df.get("Start Current(A)", np.nan)), errors="coerce")
    return out

def read_step_data(filepath):
    fmt = detect_format(filepath)
    if fmt == "nda":
        if not HAS_NDA: raise RuntimeError("NewareNDA not installed")
        rd = NewareNDA.read(filepath)
        out = rd.groupby("Step", as_index=False).agg(
            step_type=("Status","first"), v_start=("Voltage","first"),
            v_end=("Voltage","last"), i_start_mA=("Current(mA)","first"),
            chg_mAh=("Charge_Capacity(mAh)","max"), dch_mAh=("Discharge_Capacity(mAh)","max"))
        out = out.rename(columns={"Step":"Step Number"})
        out["Capacity(Ah)"]        = (out["chg_mAh"].fillna(0) + out["dch_mAh"].fillna(0)) / 1000.0
        out["Oneset Volt.(V)"]     = out["v_start"]; out["End Voltage(V)"] = out["v_end"]
        out["Starting current(A)"] = out["i_start_mA"] / 1000.0; out["Step Type"] = out["step_type"]
        return out[["Step Number","Step Type","Capacity(Ah)","Oneset Volt.(V)","End Voltage(V)","Starting current(A)"]]

    if fmt in ("xls", "xlsx"):
        engine = "xlrd" if fmt == "xls" else "openpyxl"
        try: xl = pd.ExcelFile(filepath, engine=engine)
        except ImportError as e: raise RuntimeError(f"For .xls run `pip install xlrd` ({e})")
        sheets = xl.sheet_names
        candidates = []
        for s in sheets:
            if s.lower() == "step": candidates.insert(0, s)
            elif "stat" in s.lower(): candidates.append(s)
            elif "step" in s.lower(): candidates.append(s)
        if 2 < len(sheets): candidates.append(sheets[2])
        candidates.append(sheets[0])
        last_err = None
        for s in candidates:
            try:
                df = xl.parse(s); df.columns = [str(c).strip() for c in df.columns]
                if ("Capacity(Ah)" in df.columns) and ("Step Number" in df.columns or "Step Index" in df.columns):
                    return _normalize_nw_export(df)
                if ("End Voltage(V)" in df.columns) and ("Discharge_Capacity(mAh)" in df.columns or "Charge_Capacity(mAh)" in df.columns):
                    return _normalize_neware_old(df)
            except Exception as e:
                last_err = e; continue
        raise ValueError(f"No recognised step sheet in {os.path.basename(filepath)}. Tried {candidates}. {last_err}")
    raise ValueError(f"Unsupported file: {filepath}")

def _calc_dcir(df, rest_step, pulse_step):
    pulse = df[df["Step Number"] == pulse_step]; rest = df[df["Step Number"] == rest_step]
    if pulse.empty or rest.empty: return np.nan, np.nan, np.nan
    i = float(pulse["Starting current(A)"].iloc[0])
    if i == 0 or pd.isna(i): return np.nan, np.nan, np.nan
    vr = float(rest["End Voltage(V)"].iloc[0])
    v0 = float(pulse["Oneset Volt.(V)"].iloc[0]); v30 = float(pulse["End Voltage(V)"].iloc[0])
    d0 = round(1000*(v0-vr)/i, 2); d30 = round(1000*(v30-vr)/i, 2)
    return d0, d30, round(d30-d0, 2)

def extract_formation(fp):
    df = read_step_data(fp); s = step_idx["for"]
    cap_row = df[df["Step Number"] == s["cap_full_disch_step"]]
    cap = float(cap_row["Capacity(Ah)"].iloc[0]) if not cap_row.empty else np.nan
    d0,d30,dt = _calc_dcir(df, s["rest_before_dcir"], s["dcir_pulse_step"])
    return {"FOR_discharg_cap (Ah)":cap,"FOR_dcir0 (mOhms)":d0,"FOR_dcir30 (-1C) (mOhms)":d30,"FOR_dcirt (mOhms)":dt}

def extract_before(fp):
    df = read_step_data(fp); s = step_idx["before"]
    disch = df[df["Step Number"] == s["cap_full_disch_step"]]
    chg   = df[df["Step Number"] == s["cap_full_chg_step"]]
    cd = float(disch["Capacity(Ah)"].iloc[0]) if not disch.empty else np.nan
    cc = float(chg["Capacity(Ah)"].iloc[0]) if not chg.empty else np.nan
    d0,d30,dt = _calc_dcir(df, s["rest_before_dcir"], s["dcir_pulse_step"])
    return {"Before_Storage_Discharge_Capacity (Ah)":cd, "Before_Storage_Last_Charge_Capacity (Ah)":cc,
            "Before_Storage_dcir0 (mOhms)":d0,"Before_Storage_dcir30 (-1C) (mOhms)":d30,"Before_Storage_dcirt (mOhms)":dt}

def extract_after(fp, prefix):
    df = read_step_data(fp); s = step_idx["after"]
    def _cap(step):
        r = df[df["Step Number"] == step]
        return float(r["Capacity(Ah)"].iloc[0]) if not r.empty else np.nan
    c1d = _cap(s["cap_cyc1_disch_step"]); c2c = _cap(s["cap_cyc2_chg_step"]); c2d = _cap(s["cap_cyc2_disch_step"])
    d0,d30,dt = _calc_dcir(df, s["rest_before_dcir"], s["dcir_pulse_step"])
    pfx = "After_2Weeks_Storage" if "2Weeks" in prefix else "After_1Month_Storage"
    return {f"{prefix}_Cycle1_Discharge_Capacity (Ah)":c1d,
            f"{prefix}_Cycle2_Charge_Capacity (Ah)":c2c,
            f"{prefix}_Cycle2_Discharge_Capacity (Ah)":c2d,
            f"{pfx}_dcir0 (mOhms)":d0,f"{pfx}_dcir30 (-1C) (mOhms)":d30,f"{pfx}_dcirt (mOhms)":dt}

def list_data_files(folder):
    if not folder or not os.path.isdir(folder): return []
    return [os.path.join(folder, f) for f in sorted(os.listdir(folder)) if detect_format(f) is not None]

def get_barcode(path):
    return os.path.splitext(os.path.basename(path))[0].split("_")[0]

print("Helpers ready.")


# 4. Process

In [ ]:
def _process_dir(label, folder, extractor, **extra):
    if not folder or not os.path.isdir(folder):
        print(f"  ({label}: not provided, skipping)"); return None
    files = list_data_files(folder)
    if not files: print(f"  ({label}: no data files)"); return None
    print(f"  {label}: {len(files)} files")
    rows = []
    for fp in files:
        bc = get_barcode(fp)
        try:
            feats = extractor(fp, **extra) if extra else extractor(fp)
            rows.append({"Cell Barcode": bc, **feats})
        except Exception as e:
            print(f"    skip {os.path.basename(fp)}: {e}")
    return pd.DataFrame(rows)

print("=== Processing ===")
df_for = _process_dir("Formation",     t_for.value.strip(), extract_formation)
df_bef = _process_dir("Before Storage", t_bef.value.strip(), extract_before)
df_2w  = _process_dir("2 Weeks",       t_2w.value.strip(),  extract_after, prefix="After_2Weeks_Storage")
df_1m  = _process_dir("1 Month",       t_1m.value.strip(),  extract_after, prefix="After_1month_Storage")
print()
for label, df in [("formation",df_for),("before",df_bef),("2w",df_2w),("1m",df_1m)]:
    print(f"  {label:10s}: {df.shape if df is not None else None}")


# 5. Merge + compute percentage features + optional thickness/ACR

Outer-merge all stages, compute percentage features, and merge the thickness workbook when provided.

In [ ]:
# ---------- merge all stages ----------
def _merge_pair(a, b):
    if a is None: return b
    if b is None: return a
    return a.merge(b, on="Cell Barcode", how="outer")

summary = None
for d in (df_for, df_bef, df_2w, df_1m):
    summary = _merge_pair(summary, d)

if summary is None or summary.empty:
    raise RuntimeError("No data — check pickers in Cell 2.")

print(f"merged: {summary.shape}")

# ---------- attach EL ID / Group from metadata xlsx (optional) ----------
meta_path = t_meta.value.strip()
if meta_path and os.path.isfile(meta_path):
    meta = pd.read_excel(meta_path)
    meta.columns = [str(c).strip() for c in meta.columns]
    bc_col = next((c for c in meta.columns if "barcode" in c.lower()), None)
    el_col = next((c for c in meta.columns if c.lower() in ("el id","electrolyte","group","el")), None)
    if bc_col and el_col:
        summary = summary.merge(meta[[bc_col, el_col]].rename(columns={bc_col:"Cell Barcode", el_col:"EL ID"}),
                                on="Cell Barcode", how="left")
        print(f"metadata merged on {bc_col} -> EL ID = {el_col}")
    else:
        print(f"metadata xlsx columns: {list(meta.columns)} — Barcode/EL ID not found; skipping")
if "EL ID" not in summary.columns:
    # fallback: derive lot from barcode (chars 6..7 of CELL0XXYYY)
    summary["EL ID"] = summary["Cell Barcode"].astype(str).str.extract(r"^CELL0(\d{2})\d{3}$")[0].fillna("")

# ---------- % features (capacity / DCIR) ----------
def _safe_div(a, b): return 100.0 * a / b
def _grow_pct(after, before): return 100.0 * (after - before) / before

if "After_2Weeks_Storage_Cycle1_Discharge_Capacity (Ah)" in summary.columns:
    summary["Remaining capacity% after 2 weeks"] = _safe_div(
        summary["After_2Weeks_Storage_Cycle1_Discharge_Capacity (Ah)"],
        summary["Before_Storage_Discharge_Capacity (Ah)"])
if "After_1month_Storage_Cycle1_Discharge_Capacity (Ah)" in summary.columns:
    summary["Remaining capacity% after 1 month"] = _safe_div(
        summary["After_1month_Storage_Cycle1_Discharge_Capacity (Ah)"],
        summary["Before_Storage_Discharge_Capacity (Ah)"])
if "After_2Weeks_Storage_Cycle2_Discharge_Capacity (Ah)" in summary.columns:
    summary["Recovered capacity% after 2 weeks"] = _safe_div(
        summary["After_2Weeks_Storage_Cycle2_Discharge_Capacity (Ah)"],
        summary["Before_Storage_Discharge_Capacity (Ah)"])
if "After_1month_Storage_Cycle2_Discharge_Capacity (Ah)" in summary.columns:
    summary["Recovered capacity% after 1 month"] = _safe_div(
        summary["After_1month_Storage_Cycle2_Discharge_Capacity (Ah)"],
        summary["Before_Storage_Discharge_Capacity (Ah)"])
if "After_2Weeks_Storage_dcir0 (mOhms)" in summary.columns:
    summary["DCIR Growth (%) after 2 weeks"] = _grow_pct(
        summary["After_2Weeks_Storage_dcir0 (mOhms)"],
        summary["Before_Storage_dcir0 (mOhms)"])
if "After_1Month_Storage_dcir0 (mOhms)" in summary.columns:
    summary["DCIR Growth (%) after 1 month"] = _grow_pct(
        summary["After_1Month_Storage_dcir0 (mOhms)"],
        summary["Before_Storage_dcir0 (mOhms)"])

# ---------- (OPTIONAL) thickness / ACR xlsx ----------
thick_path = t_thick.value.strip()
def _read_thickness(path):
    """Multi-section xlsx: Before / 2-week / 4-week × {Avg terrace thickness, ACR}."""
    if not path or not os.path.isfile(path):
        return None
    raw = pd.read_excel(path, sheet_name=0, header=[0,1])
    bc_col = None
    for c in raw.columns:
        if "barcode" in str(c[1]).lower() or "barcode" in str(c[0]).lower():
            bc_col = c; break
    if bc_col is None:
        print("  thickness: Cell Barcode column not found — skipping"); return None
    def grab(top_kw, sub_kw):
        for c in raw.columns:
            top = str(c[0]).lower(); sub = str(c[1]).lower()
            if any(t in top for t in top_kw) and any(s in sub for s in sub_kw):
                return c
        return None
    PHASE = {"before":["before"], "2w":["2 weeks","2week","2 wk","week 2"],
             "4w":["4 weeks","4week","4 wk","week 4"]}
    THICK = ["avg terrace thickness","avg thickness","average terrace","avg terrace"]
    ACR   = ["acr"]
    out = pd.DataFrame({"Cell Barcode": raw[bc_col].astype(str).str.strip()})
    for ph, top_kw in PHASE.items():
        ct = grab(top_kw, THICK); ca = grab(top_kw, ACR)
        out[f"thickness_avg_{ph}"] = pd.to_numeric(raw[ct], errors="coerce") if ct is not None else np.nan
        out[f"ACR_{ph}"]           = pd.to_numeric(raw[ca], errors="coerce") if ca is not None else np.nan
    out["thickness growth (%) after 2 weeks"] = _grow_pct(out["thickness_avg_2w"], out["thickness_avg_before"])
    out["thickness growth (%) after 1 month"] = _grow_pct(out["thickness_avg_4w"], out["thickness_avg_before"])
    out["ACR Growth (%) after 2 weeks"]       = _grow_pct(out["ACR_2w"],           out["ACR_before"])
    out["ACR Growth (%) after 1 month"]       = _grow_pct(out["ACR_4w"],           out["ACR_before"])
    return out

thick_df = _read_thickness(thick_path)
if thick_df is not None:
    print(f"  thickness: parsed {len(thick_df)} barcodes from {os.path.basename(thick_path)}")
    summary = summary.merge(thick_df, on="Cell Barcode", how="left")
else:
    print("  thickness: not provided -> thickness/ACR features will be NaN")
    for c in ["thickness growth (%) after 2 weeks","thickness growth (%) after 1 month",
              "ACR Growth (%) after 2 weeks","ACR Growth (%) after 1 month"]:
        if c not in summary.columns:
            summary[c] = np.nan

# Reorder so Cell Barcode + EL ID are first
front = ["Cell Barcode", "EL ID"]
rest  = [c for c in summary.columns if c not in front]
summary = summary[front + rest].sort_values(["EL ID","Cell Barcode"]).reset_index(drop=True)
display(summary.head(10))
print(f"\nTotal columns: {len(summary.columns)}, rows: {len(summary)}")


# 6. Save xlsx (with charts) + stage1ready.pkl

Charts emitted as **chartsheets** (own tab, in front). Single series + per-point colors so x-axis labels show every barcode/EL ID. Thickness / ACR charts only emitted if thickness xlsx provided.

In [ ]:
import xlsxwriter, pickle

def _palette():
    return ["#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd","#8c564b","#e377c2",
            "#7f7f7f","#bcbd22","#17becf","#aec7e8","#ffbb78","#98df8a","#ff9896",
            "#c5b0d5","#c49c94","#f7b6d2","#dbdb8d","#9edae5","#393b79"]

def _bc_color_factory(barcodes):
    p = _palette()
    lots = sorted({str(b)[5:7] for b in barcodes if isinstance(b,str) and len(b)>=7})
    lot2 = {lot: p[i % len(p)] for i,lot in enumerate(lots)}
    def f(bc):
        s = str(bc)
        return lot2.get(s[5:7] if len(s)>=7 else "", "#1f77b4")
    return f

def _el_color_factory(els):
    p = _palette()
    uniq = sorted({str(e) for e in els if str(e) and str(e) != 'nan'})
    return {e: p[i % len(p)] for i,e in enumerate(uniq)}

# (col_name, short_label) — only emitted if column has at least one non-NaN
CHART_COLS = [
    ("Remaining capacity% after 2 weeks",   "RemCap_2w"),
    ("Remaining capacity% after 1 month",   "RemCap_4w"),
    ("Recovered capacity% after 2 weeks",   "RecCap_2w"),
    ("Recovered capacity% after 1 month",   "RecCap_4w"),
    ("DCIR Growth (%) after 2 weeks",       "DCIR_2w"),
    ("DCIR Growth (%) after 1 month",       "DCIR_4w"),
    ("thickness growth (%) after 2 weeks",  "Thick_2w"),
    ("thickness growth (%) after 1 month",  "Thick_4w"),
    ("ACR Growth (%) after 2 weeks",        "ACR_2w"),
    ("ACR Growth (%) after 1 month",        "ACR_4w"),
]

# Per-lot avg/std — restrict to numeric columns (EL ID is the group key, Cell Barcode is string)
_num_cols = summary.select_dtypes(include=[np.number]).columns.tolist()
per_lot = (summary[["EL ID"] + _num_cols]
           .groupby("EL ID", dropna=False)
           .agg(["mean","std"])
           .reset_index())
per_lot.columns = ["EL ID"] + [f"{a}_{b}" for a,b in per_lot.columns[1:]]

out_dir = t_out_dir.value.strip() or os.path.dirname(t_bef.value.strip() or t_2w.value.strip() or ".")
fname = t_fname.value.strip() or "HT_storage_output.xlsx"
out_xlsx = os.path.join(out_dir, fname)
os.makedirs(out_dir, exist_ok=True)

wb = xlsxwriter.Workbook(out_xlsx)

# Pre-create chartsheets first → they appear at the FRONT of the workbook
chartsheets_lot  = {}
chartsheets_cell = {}
for col, lbl in CHART_COLS:
    if col not in summary.columns: continue
    if summary[col].notna().sum() == 0: continue   # skip if all NaN
    chartsheets_lot[col]  = wb.add_chartsheet(f"PerLot_{lbl}"[:31])
    chartsheets_cell[col] = wb.add_chartsheet(f"PerCell_{lbl}"[:31])

# Data sheets
ws_sum = wb.add_worksheet("Summary")
ws_lot = wb.add_worksheet("Per_Lot_Avg")
bold = wb.add_format({"bold": True})

def _write(ws, df):
    for j, c in enumerate(df.columns):
        ws.write(0, j, str(c), bold)
    for i, row in enumerate(df.itertuples(index=False), start=1):
        for j, v in enumerate(row):
            if pd.isna(v): continue
            if isinstance(v, (int,float,np.floating,np.integer)):
                ws.write_number(i, j, float(v))
            else:
                ws.write(i, j, str(v))

_write(ws_sum, summary)
_write(ws_lot, per_lot)

bc_color   = _bc_color_factory(list(summary["Cell Barcode"]))
el_palette = _el_color_factory(list(per_lot["EL ID"]))

barcode_idx = list(summary.columns).index("Cell Barcode")
elid_idx    = list(per_lot.columns).index("EL ID")
n_cells = len(summary); n_lots = len(per_lot)

for col, lbl in CHART_COLS:
    if col not in chartsheets_lot: continue
    col_idx_sum = list(summary.columns).index(col)
    avg_col = f"{col}_mean"
    if avg_col not in per_lot.columns: continue
    avg_idx = list(per_lot.columns).index(avg_col)

    # Per-cell chart: single series, per-point colors -> all barcodes on x-axis
    ch = wb.add_chart({"type":"column"})
    cell_pts = [{"fill":{"color":bc_color(bc)}, "border":{"color":bc_color(bc)}}
                for bc in summary["Cell Barcode"]]
    ch.add_series({
        "categories": ["Summary", 1, barcode_idx, n_cells, barcode_idx],
        "values":     ["Summary", 1, col_idx_sum, n_cells, col_idx_sum],
        "name": col,
        "points": cell_pts,
    })
    ch.set_title({"name": col, "name_font":{"name":"Helvetica Neue","size":14,"bold":True}})
    ch.set_x_axis({"num_font":{"name":"Helvetica Neue","size":10,"rotation":-45}})
    ch.set_y_axis({"name":col,"name_font":{"name":"Helvetica Neue","size":12,"bold":True}})
    ch.set_legend({"none":True})
    chartsheets_cell[col].set_chart(ch)

    # Per-lot chart: single series, per-point colors, data labels
    ch2 = wb.add_chart({"type":"column"})
    lot_pts = [{"fill":{"color":el_palette.get(str(e),"#1f77b4")},
                "border":{"color":el_palette.get(str(e),"#1f77b4")}}
               for e in per_lot["EL ID"]]
    ch2.add_series({
        "categories": ["Per_Lot_Avg", 1, elid_idx, n_lots, elid_idx],
        "values":     ["Per_Lot_Avg", 1, avg_idx,  n_lots, avg_idx],
        "name": f"{col} (lot avg)",
        "points": lot_pts,
        "data_labels":{"value":True,"num_format":"0.0"},
    })
    ch2.set_title({"name": f"{col}  (per-lot avg)",
                   "name_font":{"name":"Helvetica Neue","size":14,"bold":True}})
    ch2.set_x_axis({"num_font":{"name":"Helvetica Neue","size":11,"rotation":-30}})
    ch2.set_y_axis({"name":col,"name_font":{"name":"Helvetica Neue","size":12,"bold":True}})
    ch2.set_legend({"none":True})
    chartsheets_lot[col].set_chart(ch2)

wb.close()
print(f"saved: {out_xlsx}")

# ---- stage1ready.pkl for Stage-1 fast path ----
pkl_path = out_xlsx.replace(".xlsx", ".stage1ready.pkl")
payload = {
    "format": "ht_storage_v3",
    "ht_storage_summary": summary.copy(),
    "thickness_present": (summary["thickness growth (%) after 1 month"].notna().sum() > 0
                          if "thickness growth (%) after 1 month" in summary.columns else False),
}
with open(pkl_path, "wb") as f:
    pickle.dump(payload, f)
print(f"saved: {pkl_path}")
